# Imbalanced Data and Cost-Sensitive Classification

Tuesday's lecture gave us the tools to *visualize* the precision-recall and ROC tradeoffs. Today we put dollar amounts on those tradeoffs.

This is the payoff lecture for the classification unit. We'll see how class weights, threshold tuning, and probability calibration work together to optimize for business-relevant costs rather than accuracy. The ideas connect directly to the Elkan paper (HW3b) and the work you'll do in HW3c.

## Setup

In [ ]:
%matplotlib inline

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.calibration import CalibrationDisplay, CalibratedClassifierCV
from sklearn.datasets import make_classification
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
    make_scorer,
    precision_score,
    PrecisionRecallDisplay,
    recall_score,
)
from sklearn.model_selection import (
    cross_val_predict,
    StratifiedKFold,
    train_test_split,
    TunedThresholdClassifierCV,
)
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC

## Fraud Detection Dataset

We'll use a synthetic fraud detection dataset. Fraud is the minority class (~5%), and fraudulent transactions have higher dollar amounts on average - this gives us a natural cost structure.

In [ ]:
# Generate synthetic fraud data
X, y = make_classification(
    n_samples=10000, n_features=10, n_informative=5, n_redundant=3,
    n_classes=2, weights=[0.95, 0.05], random_state=42
)

# Add a transaction amount feature (higher for fraud)
rng = np.random.RandomState(42)
amounts = rng.exponential(scale=100, size=len(y))
amounts[y == 1] = amounts[y == 1] * 5  # fraudulent transactions are larger
X = np.column_stack([X, amounts])

feature_names = [f"feature_{i}" for i in range(10)] + ["transaction_amount"]

print(f"Samples: {len(y):,}")
print(f"Fraud rate: {y.mean():.1%}")
print(f"Fraud count: {y.sum()}")

In [ ]:
# Quick look at the data
df = pd.DataFrame(X, columns=feature_names)
df["fraud"] = y
df.head()

In [ ]:
# Transaction amounts are much larger for fraud
fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(df.loc[df["fraud"] == 0, "transaction_amount"], bins=50, alpha=0.5,
        label="Legitimate", color="steelblue")
ax.hist(df.loc[df["fraud"] == 1, "transaction_amount"], bins=50, alpha=0.5,
        label="Fraud", color="tomato")
ax.set_xlabel("Transaction Amount ($)")
ax.set_ylabel("Count")
ax.set_title("Transaction Amount Distribution by Class")
ax.legend()
plt.tight_layout()
plt.show()

print(f"Legitimate mean: ${df.loc[df['fraud']==0, 'transaction_amount'].mean():.0f}")
print(f"Fraud mean:      ${df.loc[df['fraud']==1, 'transaction_amount'].mean():.0f}")

## Why `stratify` Matters

You saw `stratify=y` in 09b and used it in HW3a. Here's why it matters even more with severe imbalance. With only 5% fraud, a random split could concentrate fraud cases unevenly between train and test:

In [ ]:
# Without stratify: fraud rate can vary significantly between splits
fraud_rates = []
for i in range(20):
    _, _, _, y_test_i = train_test_split(X, y, test_size=0.3, random_state=i)
    fraud_rates.append(y_test_i.mean())

print("Fraud rate in test set across 20 random splits (no stratify):")
print(f"  Min:  {min(fraud_rates):.1%}")
print(f"  Max:  {max(fraud_rates):.1%}")
print(f"  Mean: {np.mean(fraud_rates):.1%}")
print(f"  STD: {np.std(fraud_rates):.1%}")

In [ ]:
# With stratify: fraud rate is preserved exactly
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

print(f"Training: {len(y_train):,} samples ({y_train.mean():.1%} fraud)")
print(f"Test:     {len(y_test):,} samples ({y_test.mean():.1%} fraud)")

`stratify=y` ensures both sets have the same fraud rate as the original data. For rare events, this isn't optional - it's a correctness requirement. The same logic applies to cross-validation: `StratifiedKFold` ensures every fold has fraud cases to evaluate against.

## Baseline: Accuracy Lies

This is the third time we've seen this lesson (08b Default dataset, 09b MNIST DummyClassifier). Now with real costs.

In [ ]:
# DummyClassifier: predict the majority class every time
dummy = DummyClassifier(strategy="most_frequent")
dummy.fit(X_train, y_train)
y_pred_dummy = dummy.predict(X_test)

print(f"DummyClassifier accuracy:\t{accuracy_score(y_test, y_pred_dummy):.1%}")
print(f"Fraud detection rate (recall):\t{recall_score(y_test, y_pred_dummy):.1%}")

The fraud detection rate is recall for class 1 - also known as the True Positive Rate (TPR) from Tuesday's ROC discussion. Since the DummyClassifier always predicts "legitimate" (the majority class), it never predicts a positive (fraud), so TPR = 0. High accuracy, zero usefulness.

In [ ]:
# LogReg baseline: high accuracy, low fraud recall
baseline = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))
baseline.fit(X_train, y_train)
y_pred_base = baseline.predict(X_test)

print(f"LogReg accuracy: {accuracy_score(y_test, y_pred_base):.1%}")
print(f"\n{classification_report(y_test, y_pred_base)}")

Interpretation:

- Class 0 (not-fraud): near-perfect precision (0.97) and recall (1.00). The model almost never misclassifies a legitimate transaction.
- Class 1 (fraud): precision is decent (0.85 - when it flags fraud, it's usually right) but recall is low (0.40 - it catches less than half of actual fraud cases).
- The 163 fraud cases in the test set are vastly outnumbered by 2,837 legitimate ones. Getting the majority class right dominates accuracy.
- Macro avg recall (0.70) reveals the imbalance that accuracy (0.96) hides - the model is mediocre at the task that matters most.

A note on the last three rows of the report: *accuracy* appears aligned with the f1-score column, but it's a single overall metric - not an f1-score. *Macro avg* is the simple average of each metric across both classes (treating each class equally). *Weighted avg* weights each class by its support (number of samples), so class 0 with 2,837 samples dominates. For imbalanced data, macro avg is more revealing because it gives the minority class equal voice.

In [ ]:
# Normalized confusion matrix: recall is on the diagonal
fig, ax = plt.subplots(figsize=(6, 5))
ConfusionMatrixDisplay.from_estimator(
    baseline, X_test, y_test, normalize="true", values_format=".2f", ax=ax
)
ax.set_title("Baseline LogReg (Normalized)")
plt.tight_layout()
plt.show()

High accuracy, but look at the diagonal for class 1 (fraud). The model catches only 40% of actual fraud cases. A manager who sees "96% accuracy" would be impressed - until they learn the model is missing most of the fraud.

The problem is this - how to build more effective models when the event of interest is not well represented in the data? This connects to a fundamental idea from information theory: rare events carry more information than common ones (Shannon, 1948). That's exactly why they're both valuable to predict and difficult to learn from - the training data contains very few examples of the thing we most want to detect.

## Resampling: A Common (But Often Wrong) Fix

The most intuitive response to class imbalance is to "fix" the data itself:

- *Oversampling* duplicates minority-class samples to balance the training set. Risk: duplicating real samples leads to overfitting.
- *Undersampling* removes majority-class samples. Risk: throwing away data that could help the model.

Both approaches change the training distribution, which can distort probability estimates and shift the decision boundary in unpredictable ways.

sklearn deliberately does not include resampling tools - a design choice that says something about where the field has landed on this topic. The `imbalanced-learn` library fills this gap with an API that follows sklearn's conventions:

In [ ]:
from imblearn.over_sampling import RandomOverSampler, SMOTE
from imblearn.under_sampling import RandomUnderSampler
from imblearn.pipeline import Pipeline as ImbPipeline

In [ ]:
# Random oversampling: duplicate minority samples to match majority
ros = RandomOverSampler(random_state=42)

# the fit_resample method generates additional observations based on imbalance
X_over, y_over = ros.fit_resample(X_train, y_train)

print(f"Original:     {len(y_train):,} samples ({y_train.mean():.1%} fraud)")
print(f"Oversampled:  {len(y_over):,} samples ({y_over.mean():.1%} fraud)")

In [ ]:
# Random undersampling: remove majority samples to match minority
rus = RandomUnderSampler(random_state=42)
X_under, y_under = rus.fit_resample(X_train, y_train)

print(f"Undersampled: {len(y_under):,} samples ({y_under.mean():.1%} fraud)")
print(f"  (lost {len(y_train) - len(y_under):,} samples)")

In [ ]:
# Train on each resampled set and compare
over_model = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))
over_model.fit(X_over, y_over)
y_pred_over = over_model.predict(X_test)

under_model = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))
under_model.fit(X_under, y_under)
y_pred_under = under_model.predict(X_test)

print("=== Oversampled ===")
print(f"Accuracy: {accuracy_score(y_test, y_pred_over):.1%}  "
      f"Precision: {precision_score(y_test, y_pred_over):.1%}  "
      f"Recall: {recall_score(y_test, y_pred_over):.1%}")

print("\n=== Undersampled ===")
print(f"Accuracy: {accuracy_score(y_test, y_pred_under):.1%}  "
      f"Precision: {precision_score(y_test, y_pred_under):.1%}  "
      f"Recall: {recall_score(y_test, y_pred_under):.1%}")

Resampling improves fraud recall, but at a cost to both accuracy and precision - the model triggers more false alarms to catch more fraud. And we had to resample as a separate step before fitting, which is error-prone - if you accidentally resample before splitting, you get severe data leakage.

### SMOTE

SMOTE (Synthetic Minority Oversampling Technique) goes further: instead of duplicating existing minority samples, it generates *synthetic* ones by interpolating between neighboring minority-class points in feature space.

`imbalanced-learn` provides `ImbPipeline`, which handles resampling inside the pipeline so it only applies during `fit`, not during `predict` - avoiding leakage automatically:

In [ ]:
# SMOTE inside an imbalanced-learn Pipeline
smote_pipe = ImbPipeline([
    ("scaler", StandardScaler()),
    ("smote", SMOTE(random_state=42)),
    ("clf", LogisticRegression(max_iter=1000)),
])
smote_pipe.fit(X_train, y_train)
y_pred_smote = smote_pipe.predict(X_test)

print("=== SMOTE ===")
print(f"Accuracy: {accuracy_score(y_test, y_pred_smote):.1%}  "
      f"Precision: {precision_score(y_test, y_pred_smote):.1%}  "
      f"Recall: {recall_score(y_test, y_pred_smote):.1%}")

SMOTE is widely taught and widely used, but recent research (2024-2025) suggests its benefits are narrower than commonly believed:

- Strong learners (gradient boosting, random forests) handle imbalance well natively; SMOTE doesn't meaningfully improve them
- In high-dimensional data, SMOTE doesn't reduce majority-class bias effectively
- Simpler approaches (random undersampling, class weights) often match or beat SMOTE

<div class="alert alert-info">
  <b>⚠️ Use Sampling Methods with Caution</b><br>
    By now, it should be intuitive that duplicating, eliminating, or fabricating training data is something to approach carefully. These are established techniques with legitimate uses, but anytime you alter the distribution your model learns from, you're making assumptions about what the "right" distribution looks like. If those assumptions are wrong, you can make things worse.
  </div>

There's a simpler way to get the same effect without touching the data at all.

## Class Weights: A Better First Tool

Instead of changing the data, we can change how the model treats errors. `class_weight='balanced'` automatically upweights the minority class in proportion to its rarity - the loss function penalizes fraud misclassifications more heavily. This achieves a similar effect to oversampling but without data duplication, without leakage risk, and with no extra complexity.

In [ ]:
# Same pipeline, one parameter added
balanced = make_pipeline(
    StandardScaler(),
    LogisticRegression(max_iter=1000, class_weight="balanced")
)
balanced.fit(X_train, y_train)
y_pred_bal = balanced.predict(X_test)

print(f"Balanced accuracy: {accuracy_score(y_test, y_pred_bal):.1%}")
print(f"\n{classification_report(y_test, y_pred_bal)}")

The difference between sampling methods and class weight adjustment is subtle but important. Sampling methods change the data, weights modify the loss function math - the optimization objective changes so that the minority class matters more. In short, where oversampling tells the model "fraud happens more often," class weights tell it "fraud matters more."

In [ ]:
# Side-by-side confusion matrices
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
ConfusionMatrixDisplay.from_estimator(baseline, X_test, y_test, ax=axes[0])
axes[0].set_title("Default")
ConfusionMatrixDisplay.from_estimator(balanced, X_test, y_test, ax=axes[1])
axes[1].set_title("Balanced Weights")
plt.tight_layout()
plt.show()

The balanced model catches more fraud (higher recall) but generates more false alarms (lower precision). Accuracy dropped. Is this better or worse? It depends entirely on what errors cost.

### Precision-Recall Comparison

The PR curve from Tuesday gives us the full picture. Here we compare the baseline and balanced models:

In [ ]:
# Out-of-fold probability estimates for both models
y_scores_base = cross_val_predict(
    baseline, X_train, y_train, cv=StratifiedKFold(5), method="predict_proba"
)[:, 1]

y_scores_bal = cross_val_predict(
    balanced, X_train, y_train, cv=StratifiedKFold(5), method="predict_proba"
)[:, 1]

fig, ax = plt.subplots(figsize=(6, 6))
PrecisionRecallDisplay.from_predictions(y_train, y_scores_base, name="Default", ax=ax)
PrecisionRecallDisplay.from_predictions(y_train, y_scores_bal, name="Balanced", ax=ax)
ax.set_title("PR Curve: Default vs Balanced Weights")
ax.legend(loc="upper right")
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

The balanced model shifts the curve - it operates in a different region of the precision-recall tradeoff. Class weights don't magically create better separation; they change *where* the model sits on the curve.

## Cost Analysis: Making Costs Explicit

This is the bridge from Tuesday's cost matrix concept to real dollars. Instead of adjusting the data (resampling) or relying on class weights alone, we assign concrete costs to each error type:

- False Negative (missed fraud): the fraud amount is lost
- False Positive (false alarm): costs a fixed investigation fee of \\$20 per case

In [ ]:
# Extract transaction amounts for the test set
test_amounts = X_test[:, -1]  # last column is transaction_amount

def compute_financial_impact(y_true, y_pred, amounts, investigation_cost=20):
    """Compute dollar impact of classification decisions."""
    # create boolean masks to identify tp, fn, and fp cases
    tp = (y_true == 1) & (y_pred == 1)  # caught fraud
    fn = (y_true == 1) & (y_pred == 0)  # missed fraud
    fp = (y_true == 0) & (y_pred == 1)  # false alarms

    saved = amounts[tp].sum()           # fraud dollars recovered
    lost = amounts[fn].sum()            # fraud dollars missed
    inv_cost = fp.sum() * investigation_cost  # investigation overhead

    return {
        "Accuracy": f"{accuracy_score(y_true, y_pred):.1%}",
        "Fraud caught": f"${saved:,.0f}",
        "Fraud missed": f"${lost:,.0f}",
        "Investigation cost": f"${inv_cost:,.0f}",
        "Net savings": f"${saved - inv_cost:,.0f}",
        "Dollar catch rate": f"{saved / (saved + lost):.1%}",
    }

In [ ]:
print("=== Default LogReg ===")
for k, v in compute_financial_impact(y_test, y_pred_base, test_amounts).items():
    print(f"  {k}: {v}")

print("\n=== Balanced LogReg ===")
for k, v in compute_financial_impact(y_test, y_pred_bal, test_amounts).items():
    print(f"  {k}: {v}")

Here, "dollar catch rate" is the fraction of total fraud *dollars* recovered - essentially accuracy measured in dollars rather than counts. A model might catch 80% of fraud cases (recall) but only 60% of fraud dollars if it misses the big ones.

The balanced model has lower accuracy but higher net savings. This is the central lesson: the less accurate model can save more money when error costs are asymmetric.

## Threshold Tuning: The Modern Answer

Class weights change how the model *trains*. Threshold tuning changes how it *decides* after training. Both shift the precision-recall tradeoff toward higher recall, and you can combine them, but they're addressing the same problem from different ends. Threshold tuning is more flexible - you can optimize for any cost function without retraining.

First, let's look at where the model's probability estimates fall for actual fraud vs. non-fraud:

In [ ]:
# Predicted probability distribution - separate panels for clarity
y_proba = baseline.predict_proba(X_test)[:, 1]

fig, axes = plt.subplots(1, 2, figsize=(14, 4), sharey=False)

# Left: full view (dominated by legitimate class)
axes[0].hist(y_proba[y_test == 0], bins=50, alpha=0.6, label="Legitimate", color="steelblue")
axes[0].hist(y_proba[y_test == 1], bins=50, alpha=0.6, label="Fraud", color="tomato")
axes[0].axvline(x=0.5, color="black", linestyle="--", label="Default threshold (0.5)")
axes[0].set_xlabel("Predicted P(fraud)")
axes[0].set_ylabel("Count")
axes[0].set_title("All Samples")
axes[0].legend()

# Right: fraud cases only (zoomed in)
axes[1].hist(y_proba[y_test == 1], bins=30, alpha=0.6, color="tomato")
axes[1].axvline(x=0.5, color="black", linestyle="--", label="Default threshold (0.5)")
axes[1].set_xlabel("Predicted P(fraud)")
axes[1].set_ylabel("Count (fraud only)")
axes[1].set_title("Fraud Cases Only")
axes[1].legend()

plt.tight_layout()
plt.show()

The right panel tells the story: many fraud cases have predicted probabilities *below* the 0.5 threshold. The model knows they're suspicious but not suspicious enough to flag at the default cutoff. Lowering the threshold would catch these.

### Custom Cost Scorer with `make_scorer`

To let sklearn find the optimal threshold automatically, we need a scoring function that measures what we actually care about: total cost.

`make_scorer` wraps any function into a scorer that sklearn's CV tools can use. sklearn always *maximizes* scores, so we negate the cost (lower cost = higher score).

In [ ]:
# Cost assumptions
cost_investigation = 20  # cost per false alarm

def cost_scorer(y_true, y_pred):
    """Total financial cost of classification errors. Lower is better."""
    # Note: make_scorer only passes y_true and y_pred, so we grab transaction
    # amounts from X_train via closure. This is a simplification - the amounts
    # may not perfectly align with CV fold indices, but the approximation is
    # close enough for threshold optimization on this dataset.
    amounts = X_train[:len(y_true), -1]
    fn = (y_true == 1) & (y_pred == 0)
    fp = (y_true == 0) & (y_pred == 1)
    total_cost = amounts[fn].sum() + fp.sum() * cost_investigation
    return -total_cost  # negate: sklearn maximizes, we want to minimize cost

custom_scorer = make_scorer(cost_scorer)

### `TunedThresholdClassifierCV`

This is the sklearn 1.5+ tool that operationalizes Elkan's argument from the paper (HW3b): learn good probability estimates, then find the optimal decision threshold via cross-validation.

In 09b we set thresholds manually with `FixedThresholdClassifier`. Here, sklearn searches for the threshold that optimizes our custom cost scorer automatically.

Note the use of `StratifiedKFold` to ensure that each fold is representative of the original class balances.

In [ ]:
# Build base pipeline, then wrap with threshold tuner
base_pipe = make_pipeline(
    StandardScaler(),
    LogisticRegression(max_iter=1000),
)

tuned = TunedThresholdClassifierCV(
    base_pipe,
    scoring=custom_scorer,
    cv=StratifiedKFold(n_splits=5),
)
tuned.fit(X_train, y_train)

print(f"Optimal threshold: {tuned.best_threshold_:.4f}")

The optimal threshold is *well* below 0.5 - the model now flags samples as fraud at a much lower probability. This makes sense given that missing fraud is far more expensive than investigating a false alarm.

In [ ]:
# Compare all three approaches
y_pred_tuned = tuned.predict(X_test)

approaches = {
    "Default (t=0.5)": y_pred_base,
    "Balanced weights": y_pred_bal,
    f"Tuned (t={tuned.best_threshold_:.3f})": y_pred_tuned,
}

rows = []
for name, y_pred in approaches.items():
    cm = confusion_matrix(y_test, y_pred)
    tp_amounts = test_amounts[(y_test == 1) & (y_pred == 1)].sum()
    fn_amounts = test_amounts[(y_test == 1) & (y_pred == 0)].sum()
    rows.append({
        "Approach": name,
        "Accuracy": f"{accuracy_score(y_test, y_pred):.1%}",
        "Recall": f"{recall_score(y_test, y_pred):.1%}",
        "FP": cm[0, 1],
        "FN": cm[1, 0],
        "Fraud caught": f"${tp_amounts:,.0f}",
        "Fraud missed": f"${fn_amounts:,.0f}",
        "Investigation": f"${cm[0, 1] * 20:,.0f}",
        "Net savings": f"${tp_amounts - cm[0, 1] * 20:,.0f}",
    })

comparison = pd.DataFrame(rows)
print(comparison.to_string(index=False))

The threshold-tuned model finds a different point on the precision-recall curve than either the default or balanced approach - one that's optimized specifically for the cost structure of this problem.

The cost-tuned model saves less (fraud caught) and costs more (fraud missed) than the balanced model, but it reduces investigation costs (fewer false alarms) enough to improve the overall net savings. It optimizes for the real business outcome instead of any single metric.

## Calibration: Can You Trust the Probabilities?

Everything we just did - threshold tuning, cost calculations, the probability histogram above - depends on `predict_proba` meaning what it says. If the model says P(fraud) = 0.3, do roughly 30% of those cases turn out to be fraud?

To check this, we group all test samples by their predicted probability and count what fraction were actually fraud in each group. For example, take all samples where the model predicted P(fraud) between 0.25 and 0.35 - if the model is calibrated, roughly 30% of that group should actually be fraud. The reliability diagram plots this for every probability bin:

In [ ]:
# Show the calibration concept with actual numbers
y_proba_test = baseline.predict_proba(X_test)[:, 1]

# Bin predictions and compute observed fraud rate per bin
bin_edges = [0.0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0]
cal_rows = []
for lo, hi in zip(bin_edges[:-1], bin_edges[1:]):
    mask = (y_proba_test >= lo) & (y_proba_test < hi)
    n = mask.sum()
    if n > 0:
        midpoint = (lo + hi) / 2
        actual_rate = y_test[mask].mean()
        cal_rows.append({
            "Predicted P(fraud)": f"{lo:.1f} - {hi:.1f}",
            "Samples": n,
            "Actual fraud": int(y_test[mask].sum()),
            "Observed rate": f"{actual_rate:.1%}",
            "Gap": f"{actual_rate - midpoint:+.1%}",
        })

cal_df = pd.DataFrame(cal_rows)
print(cal_df.to_string(index=False))

If the model is well-calibrated, the observed rate should roughly match the midpoint of each bin (gap near zero). The reliability diagram visualizes this - points on the diagonal mean perfect calibration.

In [ ]:
# Compare calibration: LogReg vs SVM on our fraud data
svm = make_pipeline(StandardScaler(), SVC(kernel="rbf", probability=True))
svm.fit(X_train, y_train)

fig, ax = plt.subplots(figsize=(6, 6))
CalibrationDisplay.from_estimator(baseline, X_test, y_test, n_bins=10, name="LogReg", ax=ax)
CalibrationDisplay.from_estimator(svm, X_test, y_test, n_bins=10, name="SVM (RBF)", ax=ax)
ax.set_title("Reliability Diagram (5% fraud)")
plt.tight_layout()
plt.show()

These curves are noisy. With only 5% positive class and 10 bins, some bins have very few fraud cases - the fraction of positives is estimated from a handful of samples. Calibration isn't necessarily *bad*; it's just hard to *verify* with rare events.

To see the concept more clearly, let's generate the same dataset with less imbalance:

In [ ]:
# Same data-generating process, less imbalance (30% positive)
X_bal, y_bal = make_classification(
    n_samples=10000, n_features=10, n_informative=5, n_redundant=3,
    n_classes=2, weights=[0.70, 0.30], random_state=42
)

X_bal_train, X_bal_test, y_bal_train, y_bal_test = train_test_split(
    X_bal, y_bal, test_size=0.3, random_state=42, stratify=y_bal
)

# Fit the same models on the balanced version
logreg_bal = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))
logreg_bal.fit(X_bal_train, y_bal_train)

svm_bal = make_pipeline(StandardScaler(), SVC(kernel="rbf", probability=True))
svm_bal.fit(X_bal_train, y_bal_train)

fig, ax = plt.subplots(figsize=(6, 6))
CalibrationDisplay.from_estimator(logreg_bal, X_bal_test, y_bal_test, n_bins=10, name="LogReg", ax=ax)
CalibrationDisplay.from_estimator(svm_bal, X_bal_test, y_bal_test, n_bins=10, name="SVM (RBF)", ax=ax)
ax.set_title("Reliability Diagram (30% positive — same data, less imbalance)")
plt.tight_layout()
plt.show()

With more positive samples per bin, the curves are smoother and the pattern is clearer. LogReg tends to hug the diagonal because it directly optimizes log-likelihood, which naturally produces calibrated probabilities. SVM's probability estimates (generated via Platt scaling) are typically less reliable.

### Fixing Miscalibration with `CalibratedClassifierCV`

When probabilities aren't well-calibrated, `CalibratedClassifierCV` can fix them. It wraps any classifier and uses cross-validation to learn a calibration mapping (sigmoid or isotonic) from raw scores to calibrated probabilities.

In [ ]:
# Calibrate the SVM on the 30% dataset where the effect is visible
svm_calibrated = CalibratedClassifierCV(svm_bal, cv=5, method="sigmoid")
svm_calibrated.fit(X_bal_train, y_bal_train)

fig, ax = plt.subplots(figsize=(6, 6))
CalibrationDisplay.from_estimator(svm_bal, X_bal_test, y_bal_test, n_bins=10, name="SVM (uncalibrated)", ax=ax)
CalibrationDisplay.from_estimator(svm_calibrated, X_bal_test, y_bal_test, n_bins=10, name="SVM (calibrated)", ax=ax)
CalibrationDisplay.from_estimator(logreg_bal, X_bal_test, y_bal_test, n_bins=10, name="LogReg", ax=ax)
ax.set_title("Calibration: Before and After")
plt.tight_layout()
plt.show()

### When Does Calibration Matter?

Calibration matters when you're using probability *values* to make decisions - not just rankings:

- *Threshold tuning* assumes the probabilities are on a meaningful scale. A threshold of 0.3 only makes sense if 0.3 means "30% chance."
- *Expected cost calculations* like P(fraud) $\times$ fraud_amount require calibrated probabilities to produce meaningful dollar estimates.
- *Risk scoring* in regulated industries (credit, insurance, healthcare) often requires that predicted probabilities reflect real-world frequencies.

If you only care about *ranking* - which model is better, which samples are most likely positive - calibration doesn't matter. AUC captures ranking quality regardless of calibration.

This matters because:
- LogReg is naturally well-calibrated (it optimizes log-likelihood directly)
- SVM probabilities come from Platt scaling and are often poorly calibrated
- Tree-based models (coming next unit) tend to push probabilities toward extremes or toward 0.5

Back to our fraud data: LogReg is well-calibrated by nature, so the threshold tuning we did above is trustworthy. If we were using SVM or trees on imbalanced data, we'd want to calibrate first - and we'd need to be aware that verifying calibration is harder when the positive class is rare.

## When NOT to Treat Imbalance

Not every imbalanced dataset needs intervention. Before reaching for class weights, threshold tuning, or resampling, ask:

- *Is the model actually failing on the minority class?* Check recall for the minority class. If it's already high, intervention may hurt more than it helps.
- *Is the decision boundary well-separated?* If the classes don't overlap much, the model can already find the boundary. Reweighting or resampling can *distort* a boundary that was fine.
- *Are you using a strong learner?* Ensemble methods (Random Forest, Gradient Boosting) handle imbalance natively. Adding SMOTE or aggressive class weights on top can degrade performance.

The right diagnostic sequence: fit the model, check per-class metrics, *then* decide if intervention is needed.

## Summary

- `stratify=y` in `train_test_split` and `StratifiedKFold` for CV are correctness requirements with imbalanced data, not optional enhancements.
- Resampling (over/undersampling, SMOTE) is the intuitive fix but adds complexity and risk. sklearn deliberately excludes resampling; `imbalanced-learn` fills that gap. Duplicating, removing, or fabricating training data should always give you pause.
- `class_weight='balanced'` upweights the minority class - one line of code, big impact, no data manipulation needed.
- Making costs explicit in dollars reveals that the "less accurate" model can save more money when error costs are asymmetric.
- `TunedThresholdClassifierCV` with a custom `make_scorer` finds the cost-optimal threshold via cross-validation - this is Elkan's prescription operationalized.
- Probability calibration determines whether `predict_proba` values are trustworthy for cost calculations. LogReg is naturally well-calibrated; other models may need `CalibratedClassifierCV`. Calibration is harder to verify with rare events.
- Not every imbalanced dataset needs treatment. Check per-class metrics first.